# 구조화 Sokoban 에이전트 일반화 실험

## 목적

원시 행동 LLM, 구조화 정책, 국소 탐색, 설명 제거, 기존 full guard와 bounded A* oracle을 동일 held-out 사례에서 비교한다. 정책 실행은 제품과 같은 LangGraph runner를 쓰며 A*는 실행 후 비교 기준으로만 사용한다.

## 재현성 계약

- `benchmarks/boxoban_research_v1.json`의 checksum, 공식 난이도와 구조 태그
- `SOKOBAN_PROMPT_COMMIT`의 immutable LangSmith commit
- git graph revision, Ollama 모델 설정, seed와 탐색 한도
- 개발 fixture와 test level ID의 비중복

In [ ]:
import os
import subprocess
from dataclasses import asdict

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from sokoban_agent.env import Action
from sokoban_agent.env.rules import (
    apply_action,
    initial_state,
    observation_for,
)
from sokoban_agent.evaluation import (
    ResearchRunConfig,
    load_agentic_cohort_manifest,
    run_research_experiment,
)
from sokoban_agent.planning import LLMPlanner, SearchGuardPlanner
from sokoban_agent.planning.llm import OllamaClient, OllamaSettings
from sokoban_agent.planning.llm_planner import serialize_board

pd.set_option('display.max_columns', None)
load_dotenv('.env', override=True)
prompt_commit = os.environ['SOKOBAN_PROMPT_COMMIT']
cohort = load_agentic_cohort_manifest(
    'benchmarks/boxoban_research_v1.json'
)
settings = OllamaSettings.from_env()
git_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], text=True
).strip()
config = ResearchRunConfig(
    prompt_name='sokoban-strategy',
    prompt_commit=prompt_commit,
    graph_id='sokoban_agent',
    graph_revision=git_commit,
    model_name=settings.model,
    seeds=(0, 1, 2),
    max_steps=100,
    model_config={
        'temperature': settings.temperature,
        'num_ctx': settings.num_ctx,
        'max_output_tokens': settings.max_output_tokens,
        'think': settings.think,
    },
)
config

## 동일 사례에서 6개 정책 실행

In [ ]:
primitive_client = OllamaClient(settings)
guard_client = OllamaClient(settings)
primitive = LLMPlanner(
    primitive_client, model_name=settings.model
)
full_guard = SearchGuardPlanner(
    LLMPlanner(guard_client, model_name=settings.model),
    algorithm='astar',
    fallback_policy='full',
    measure_contribution=True,
)
experiment = run_research_experiment(
    cohort,
    config,
    primitive_planner=primitive,
    full_guard_planner=full_guard,
)
records = pd.DataFrame(
    asdict(record) for record in experiment.records
)
summaries = pd.DataFrame(
    asdict(summary) for summary in experiment.summaries
)
display(pd.DataFrame([experiment.run_manifest]))
display(summaries)

## 구조별 일반화와 비용 분리

In [ ]:
display(pd.pivot_table(
    records,
    index='policy_name',
    columns='difficulty',
    values='success',
    aggfunc='mean',
))
cost_columns = [
    'policy_name', 'llm_calls', 'llm_prompt_tokens',
    'llm_output_tokens', 'llm_elapsed_seconds', 'rule_checks',
    'reachability_calls', 'local_search_calls',
    'local_expanded_states', 'algorithm_calls',
    'algorithm_expanded_states', 'policy_elapsed_seconds',
]
display(records[cost_columns].groupby('policy_name').sum())

## 설명-행동 인과 개입

문자열 유사도 대신 동일 사례에서 rationale만 제거하고, 정확한 행동 sequence와 성공 여부가 바뀌는지 비교한다.

In [ ]:
pd.DataFrame([asdict(experiment.rationale_intervention)])

## 실제 환경 trajectory 점검

In [ ]:
selected = next(
    record for record in experiment.records
    if record.policy_name == 'structured-local-search'
    and record.action_sequence
)
case = next(
    case for case in cohort.levels
    if case.level_id == selected.level_id
)
level = case.to_level()
state = initial_state(level)
trajectory = [{
    'step': 0,
    'action': None,
    'board': serialize_board(observation_for(level, state)),
}]
for step, action_name in enumerate(selected.action_sequence, 1):
    move = apply_action(level, state, Action[action_name])
    state = move.state
    trajectory.append({
        'step': step,
        'action': action_name,
        'board': serialize_board(observation_for(level, state)),
    })
pd.DataFrame(trajectory)

## 해석 제한

`current-full-guard`와 `astar-oracle`은 시스템 상한·사후 비교 기준이며 에이전트 독립 해결 성공률로 해석하지 않는다. bounded A* 실패는 해답 없음이 아니라 oracle 미해결이다.